In [3]:
import pandas as pd


# Load datasets
customers = pd.read_csv('customers.csv')
products = pd.read_csv('products.csv')
ratings = pd.read_csv('ratings.csv')
orders = pd.read_csv('orders.csv')

# convert order_date column to datetime format
orders['order_date'] = pd.to_datetime(orders['order_date'])

# merge the data
merge_data = orders.merge(products, on='product_id').merge(customers, on='customer_id')

merge_data.head()

,order_id,customer_id,product_id,order_date,quantity,product_name,price,category,name
0,1,66,38,2023-08-09 08:39:23.971834,4,Bekant Conference Table,441,Tables & Desks,Customer_66
1,2,10,29,2023-09-08 08:39:23.971834,4,Råskog Stool,855,Chairs,Customer_10
2,3,58,9,2023-07-29 08:39:23.971834,3,Micke Desk,19,Tables & Desks,Customer_58
3,4,33,44,2023-09-13 08:39:23.971834,4,Koppang Dresser,765,Decor,Customer_33
4,5,32,47,2023-07-24 08:39:23.971834,2,Nymö Lamp Shade,157,Lighting,Customer_32


In [4]:
# calculate the revenue and units sold for each product
product_performance = merge_data.groupby('product_name').agg({
    'price':'sum', # this will give us the revenue
    'quantity':'sum' # this will give us the total units sold
}).reset_index()

# sort by revenue and units sold to get top products
top_products_revenue = product_performance.sort_values(by='price', ascending=False)
top_products_units = product_performance.sort_values(by='quantity', ascending=False)

In [5]:
top_products_revenue

,product_name,price,quantity
5,Brimnes Bed Storage,23190,77
41,Småstad Wardrobe,20874,52
37,Råskog Stool,20520,59
9,Fredde Desk,18854,46
8,Fjälla Storage Box,18734,44
3,Bestå TV Bench,18400,46
27,Nockeby Sofa,18101,58
6,Docksta Table,18039,54
14,Ivar Cabinet,17862,55
12,Hemnes Daybed,16900,53


In [6]:
top_products_units

,product_name,price,quantity
22,Mackapar Shoe Storage,15114,80
5,Brimnes Bed Storage,23190,77
31,Poäng Armchair,13656,63
45,Söderhamn Sofa Section,11691,63
1,Bekant Conference Table,11907,62
24,Melltorp Dining Table,10332,62
43,Strandmon Wing Chair,2050,61
49,Valje Wall Cabinet,16056,60
7,Ektorp Sofa,14574,59
37,Råskog Stool,20520,59


In [7]:
# filter orders for the last month

last_month = merge_data[merge_data['order_date'].dt.month == merge_data['order_date'].dt.month.max()]

# calculate total purchase amount for each customer in the last month
top_clients_last_month = last_month.groupby('name').agg(({
    'price':'sum'
})).reset_index()

top_clients_last_month = top_clients_last_month.sort_values(by='price', ascending=False)

In [8]:
top_clients_last_month

,name,price
0,Customer_1,5466
26,Customer_33,4476
38,Customer_44,4259
46,Customer_52,4236
71,Customer_76,3981
...,...,...
78,Customer_82,187
3,Customer_11,184
4,Customer_12,176
56,Customer_61,157


# RFM Analysis

In [11]:
import datetime as dt


# Calculate the maximum order date to use as a reference

latest_date = merge_data['order_date'].max() + dt.timedelta(days=1)

# Calculate RFM values
rfm = merge_data.groupby('customer_id').agg({
    'order_date':lambda x: (latest_date - x.max()).days, 
    'order_id': 'count', #freq number of orders
    'price': 'sum'
}).rename(columns={
    'order_date': 'recency',
    'order_id':'frequency',
    'price':'monetary'
})

# Assign scores from 1 to 5 for each metric
rfm['recency_score'] = pd.qcut(rfm['recency'], 5, labels=[5, 4, 3, 2, 1])
rfm['frequency_score'] = pd.qcut(rfm['frequency'], 5, labels=[5, 4, 3, 2, 1])
rfm['monetary_score'] = pd.qcut(rfm['monetary'], 5, labels=[5, 4, 3, 2, 1])

rfm['rfm_score'] = rfm['recency_score'].astype(str) + rfm['frequency_score'].astype(str) + rfm['monetary_score'].astype(str)

# Displaying top customers
rfm_sorted = rfm.sort_values(by=['rfm_score'], ascending=False)


In [12]:
rfm_sorted

,recency,frequency,monetary,recency_score,frequency_score,monetary_score,rfm_score
customer_id,,,,,,,
23,2,7,3369,5,5,5,555
82,1,7,3470,5,5,5,555
77,2,8,3267,5,4,5,545
85,1,8,4367,5,4,4,544
34,2,9,4075,5,4,4,544
...,...,...,...,...,...,...,...
80,17,11,5585,1,3,3,133
9,16,11,6928,1,3,2,132
57,13,12,6619,1,2,2,122
